# LeetCode 1418 — Display Table of Food Orders in a Restaurant bằng Python

Link đề: https://leetcode.com/problems/display-table-of-food-orders-in-a-restaurant/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Sorting, Ordered Set

Skeleton LeetCode:

```python
class Solution:
    def displayTable(self, orders: List[List[str]]) -> List[List[str]]:
        ...
```

## 1. Mục tiêu học

1. Biết mô hình hoá bài "lập bảng thống kê" thành **đếm tần suất theo cặp khoá** `(bàn, món)`.
2. Thành thạo `dict` lồng nhau, `defaultdict`, `Counter` và cách lấy giá trị mặc định 0 cho món không gọi.
3. Phân biệt **sắp xếp chuỗi** (theo bảng chữ, mã ký tự) và **sắp xếp số** (`"10"` phải đứng sau `"3"`) — dùng `key=int`.
4. Đi từ cách đếm thô `O(T·F·N)` đến cách một lượt `O(N + T·F + F log F + T log T)`, và cách "bucket" theo số bàn 1..500.
5. Tránh lỗi: sort số bàn dạng chuỗi, trả số nguyên thay vì chuỗi, quên cột 0, đếm theo khách thay vì theo lượt gọi.

## 2. Đề bài & ví dụ

Cho mảng `orders`, mỗi phần tử `orders[i] = [customerName, tableNumber, foodItem]`: tên khách, **số bàn** (dạng chuỗi), **tên món**.

Trả về "**bảng hiển thị**" của nhà hàng:
- Hàng đầu (header): `"Table"` rồi đến **tên các món theo thứ tự bảng chữ cái**.
- Mỗi hàng tiếp theo ứng với một bàn: cột đầu là số bàn, các cột sau là **số lượng** mỗi món mà bàn đó đã gọi (đều ghi dạng chuỗi, món không gọi ghi `"0"`).
- Các hàng bàn sắp theo **số bàn tăng dần (theo giá trị số)**. Tên khách **không** xuất hiện trong bảng.

**Ví dụ 1:**
`orders = [["David","3","Ceviche"],["Corina","10","Beef Burrito"],["David","3","Fried Chicken"],["Carla","5","Water"],["Carla","5","Ceviche"],["Rous","3","Ceviche"]]`
→ `[["Table","Beef Burrito","Ceviche","Fried Chicken","Water"],["3","0","2","1","0"],["5","0","1","0","1"],["10","1","0","0","0"]]`

**Ví dụ 2:** `[["James","12","Fried Chicken"],["Ratesh","12","Fried Chicken"],["Amadeus","12","Fried Chicken"],["Adam","1","Canadian Waffles"],["Brianna","1","Canadian Waffles"]]`
→ `[["Table","Canadian Waffles","Fried Chicken"],["1","2","0"],["12","0","3"]]`

**Ví dụ 3:** `[["Laura","2","Bean Burrito"],["Jhon","2","Beef Burrito"],["Melissa","2","Soda"]]`
→ `[["Table","Bean Burrito","Beef Burrito","Soda"],["2","1","1","1"]]`

**Ràng buộc:** `1 <= orders.length <= 5·10^4`; tên khách, tên món dài 1..20, gồm chữ hoa/thường tiếng Anh và dấu cách; số bàn là số nguyên hợp lệ trong `[1, 500]`.

In [1]:
import random
import time
from collections import defaultdict, Counter
from typing import List

ex1 = [["David","3","Ceviche"],["Corina","10","Beef Burrito"],["David","3","Fried Chicken"],
       ["Carla","5","Water"],["Carla","5","Ceviche"],["Rous","3","Ceviche"]]
out1 = [["Table","Beef Burrito","Ceviche","Fried Chicken","Water"],["3","0","2","1","0"],
        ["5","0","1","0","1"],["10","1","0","0","0"]]
ex2 = [["James","12","Fried Chicken"],["Ratesh","12","Fried Chicken"],["Amadeus","12","Fried Chicken"],
       ["Adam","1","Canadian Waffles"],["Brianna","1","Canadian Waffles"]]
out2 = [["Table","Canadian Waffles","Fried Chicken"],["1","2","0"],["12","0","3"]]
ex3 = [["Laura","2","Bean Burrito"],["Jhon","2","Beef Burrito"],["Melissa","2","Soda"]]
out3 = [["Table","Bean Burrito","Beef Burrito","Soda"],["2","1","1","1"]]
examples = [(ex1, out1), (ex2, out2), (ex3, out3)]

def show(table):
    widths = [max(len(row[j]) for row in table) for j in range(len(table[0]))]
    for row in table:
        print(" | ".join(cell.ljust(w) for cell, w in zip(row, widths)))

show(out1)

Table | Beef Burrito | Ceviche | Fried Chicken | Water
3     | 0            | 2       | 1             | 0    
5     | 0            | 1       | 0             | 1    
10    | 1            | 0       | 0             | 0    


## 3. Kiến thức nền

### 3.1 `dict` và `dict.get(key, default)`

Từ điển ánh xạ khoá → giá trị, tra cứu trung bình `O(1)`. `d.get(k, 0)` trả 0 khi khoá chưa có — rất hợp để in "0" cho món không gọi.

In [2]:
d = {"Ceviche": 2}
d["Water"] = d.get("Water", 0) + 1
print(d, d.get("Soda", 0))

{'Ceviche': 2, 'Water': 1} 0


### 3.2 `defaultdict` và `Counter`

- `defaultdict(int)`: truy cập khoá chưa có sẽ tự tạo giá trị `0`.
- `defaultdict(Counter)`: dict lồng — `cnt[ban][mon] += 1` không cần khởi tạo.
- `Counter`: dict đếm; khoá vắng mặt trả `0` (không tạo khoá mới khi chỉ đọc).

In [3]:
cnt = defaultdict(Counter)
cnt["3"]["Ceviche"] += 1
cnt["3"]["Ceviche"] += 1
cnt["5"]["Water"] += 1
print(dict(cnt))
print(cnt["3"]["Water"], "| khoá 'Water' có bị tạo?", "Water" in cnt["3"])
pair = Counter([("3", "Ceviche"), ("3", "Ceviche"), ("5", "Water")])
print(pair[("3", "Ceviche")], pair[("10", "Soda")])

{'3': Counter({'Ceviche': 2}), '5': Counter({'Water': 1})}
0 | khoá 'Water' có bị tạo? False
2 0


### 3.3 `set` để lấy giá trị phân biệt

Tập hợp loại trùng tự động. `{o[2] for o in orders}` cho mọi tên món khác nhau.

In [4]:
foods = {o[2] for o in ex1}
tables = {o[1] for o in ex1}
print(foods, tables)

{'Fried Chicken', 'Water', 'Beef Burrito', 'Ceviche'} {'10', '3', '5'}


### 3.4 Sắp xếp chuỗi vs sắp xếp số — `sorted(..., key=int)`

So sánh chuỗi là theo **từng ký tự**: `"10" < "3"` vì `'1' < '3'`. Muốn thứ tự số phải chuyển bằng `key=int`. Lưu ý thêm: chữ **hoa** có mã nhỏ hơn chữ thường (`'Z' < 'a'`), `sorted()` mặc định so theo mã ký tự — đó cũng là cách LeetCode sắp tên món.

In [5]:
print(sorted(["3", "10", "5", "100", "2"]))
print(sorted(["3", "10", "5", "100", "2"], key=int))
print(sorted(["water", "Soda", "beef", "Ceviche"]))   # chữ hoa trước chữ thường

['10', '100', '2', '3', '5']
['2', '3', '5', '10', '100']
['Ceviche', 'Soda', 'beef', 'water']


### 3.5 List comprehension và `str()`

Mỗi hàng kết quả là list chuỗi: `[ban] + [str(cnt[ban][f]) for f in foods]`. Đề yêu cầu **chuỗi**, nên nhớ `str()`.

In [6]:
foods_sorted = sorted(foods)
row = ["3"] + [str(cnt["3"][f]) for f in foods_sorted]
print(foods_sorted, row)

['Beef Burrito', 'Ceviche', 'Fried Chicken', 'Water'] ['3', '0', '2', '0', '0']


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1. Bỏ cột tên khách, chỉ giữ `(bàn, món)`:

| lượt | bàn | món |
|---|---|---|
| 1 | 3 | Ceviche |
| 2 | 10 | Beef Burrito |
| 3 | 3 | Fried Chicken |
| 4 | 5 | Water |
| 5 | 5 | Ceviche |
| 6 | 3 | Ceviche |

Đếm theo cặp: `(3,Ceviche)=2`, `(3,Fried Chicken)=1`, `(5,Water)=1`, `(5,Ceviche)=1`, `(10,Beef Burrito)=1`.

- Cột (món) sắp theo bảng chữ: `Beef Burrito, Ceviche, Fried Chicken, Water`.
- Hàng (bàn) sắp theo số: `3, 5, 10` (chú ý `10` cuối cùng!).
- Ô trống ⇒ `0`.

Vậy bài chỉ gồm 3 việc: **đếm cặp**, **sắp hai trục**, **in lưới**.

In [7]:
pairs = [(o[1], o[2]) for o in ex1]
print(Counter(pairs))

Counter({('3', 'Ceviche'): 2, ('10', 'Beef Burrito'): 1, ('3', 'Fried Chicken'): 1, ('5', 'Water'): 1, ('5', 'Ceviche'): 1})


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** lấy tập món và tập bàn, sắp xếp; với **mỗi ô** `(bàn, món)` duyệt lại **toàn bộ** `orders` để đếm. Đơn giản, đúng, nhưng chậm.

In [8]:
def display_brute(orders):
    foods = sorted({o[2] for o in orders})
    tables = sorted({o[1] for o in orders}, key=int)
    res = [["Table"] + foods]
    for t in tables:
        row = [t]
        for f in foods:
            c = 0
            for _, tt, ff in orders:          # quét lại toàn bộ cho từng ô
                if tt == t and ff == f:
                    c += 1
            row.append(str(c))
        res.append(row)
    return res

for orders, want in examples:
    print(display_brute(orders) == want)

True
True
True


**Độ phức tạp:** gọi `N` = số lượt gọi, `T` = số bàn khác nhau (≤ 500), `F` = số món khác nhau. Mỗi ô tốn `O(N)` ⇒ `O(T · F · N)`. Trường hợp xấu `T = 500`, `F` vài trăm, `N = 5·10^4` ⇒ hàng tỉ phép so sánh ⇒ **TLE**.

**Nút thắt:** đếm lại cùng một dữ liệu `T·F` lần. Chỉ cần **đếm một lượt** và lưu vào bảng băm.

In [9]:
rng = random.Random(0)
foods_pool = [f"Food{i:02d}" for i in range(30)]
mid = [[f"c{i}", str(rng.randint(1, 60)), rng.choice(foods_pool)] for i in range(3000)]
t = time.perf_counter(); display_brute(mid); tb = time.perf_counter() - t
print(f"brute với N=3000, T≈60, F=30: {tb:.3f}s (N=5·10^4, T=500 sẽ chậm hơn hàng trăm lần)")

brute với N=3000, T≈60, F=30: 0.154s (N=5·10^4, T=500 sẽ chậm hơn hàng trăm lần)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** số lượng ở ô `(bàn, món)` chỉ phụ thuộc vào **số lần cặp đó xuất hiện**. Vì vậy:
1. Duyệt `orders` **một lần**, `cnt[bàn][món] += 1` và gom tập món.
2. **Nếu** cần thứ tự cột **thì** `sorted(foods)`; **nếu** cần thứ tự hàng **thì** `sorted(tables, key=int)`.
3. Đọc ô bằng `cnt[bàn].get(món, 0)` (hoặc `Counter` tự trả 0).

**Bất biến sau bước 1:** `cnt[t][f]` = số chỉ số `i` sao cho `orders[i][1] == t` và `orders[i][2] == f`.

Phần còn lại chỉ là trình bày. Khéo hơn nữa: số bàn ∈ `[1, 500]` ⇒ có thể dùng **mảng 501 hàng** đánh chỉ số trực tiếp bằng `int(bàn)` — hàng đã "tự sắp" theo số, không cần sort bàn (counting sort).

## 7. Các cách tiếp theo

### Cách 2 — `defaultdict(Counter)` lồng nhau + sort hai trục

**Từng đoạn:**
1. `cnt = defaultdict(Counter)`; `foods = set()`.
2. Với mỗi `[_, t, f]`: `cnt[t][f] += 1`, `foods.add(f)`.
3. `foods = sorted(foods)`; header `["Table"] + foods`.
4. Với mỗi `t` trong `sorted(cnt, key=int)`: hàng `[t] + [str(cnt[t][f]) for f in foods]`.

In [10]:
def display_nested(orders):
    cnt = defaultdict(Counter)
    foods = set()
    for _, t, f in orders:
        cnt[t][f] += 1
        foods.add(f)
    foods = sorted(foods)
    res = [["Table"] + foods]
    for t in sorted(cnt, key=int):
        res.append([t] + [str(cnt[t][f]) for f in foods])
    return res

for orders, want in examples:
    print(display_nested(orders) == want)

True
True
True


In [11]:
class SolutionNested:
    def displayTable(self, orders: List[List[str]]) -> List[List[str]]:
        cnt = defaultdict(Counter)
        foods = set()
        for _, t, f in orders:
            cnt[t][f] += 1
            foods.add(f)
        foods = sorted(foods)
        res = [["Table"] + foods]
        for t in sorted(cnt, key=int):
            res.append([t] + [str(cnt[t][f]) for f in foods])
        return res

show(SolutionNested().displayTable(ex2))

Table | Canadian Waffles | Fried Chicken
1     | 2                | 0            
12    | 0                | 3            


**Độ phức tạp:** đếm `O(N)` (giả sử băm chuỗi độ dài ≤ 20 là hằng số), sort món `O(F log F)`, sort bàn `O(T log T)`, in lưới `O(T · F)`. Bộ nhớ `O(T · F)` cho kết quả (bảng đếm chỉ lưu cặp xuất hiện: `O(min(N, T·F))`).

### Cách 3 — `Counter` trên cặp `(bàn, món)` (bảng băm phẳng)

Thay vì dict lồng, đếm thẳng các tuple. Khi in, tra `pair[(t, f)]`. Ưu điểm: một dòng đếm; nhược: phải tự gom tập bàn.

In [12]:
def display_pairs(orders):
    pair = Counter((t, f) for _, t, f in orders)
    foods = sorted({f for _, _, f in orders})
    tables = sorted({t for _, t, _ in orders}, key=int)
    return [["Table"] + foods] + [[t] + [str(pair[(t, f)]) for f in foods] for t in tables]

for orders, want in examples:
    print(display_pairs(orders) == want)

True
True
True


In [13]:
class SolutionPairs:
    def displayTable(self, orders: List[List[str]]) -> List[List[str]]:
        pair = Counter((t, f) for _, t, f in orders)
        foods = sorted({f for _, _, f in orders})
        tables = sorted({t for _, t, _ in orders}, key=int)
        return [["Table"] + foods] + [[t] + [str(pair[(t, f)]) for f in foods] for t in tables]

print(SolutionPairs().displayTable(ex3))

[['Table', 'Bean Burrito', 'Beef Burrito', 'Soda'], ['2', '1', '1', '1']]


**Độ phức tạp:** như cách 2: `O(N + F log F + T log T + T·F)`.

### Cách 4 — Mảng theo số bàn (bucket 1..500) + chỉ số món

**Từng đoạn:**
1. Sắp món trước, tạo `idx = {food: j}` để biết cột của món.
2. Tạo `grid = [[0]*F for _ in range(501)]` và `used = [False]*501`.
3. Mỗi lượt: `grid[int(t)][idx[f]] += 1`, `used[int(t)] = True`.
4. Duyệt `t` từ 1 đến 500, bàn nào `used` thì in — thứ tự số có sẵn, không cần sort bàn.

Đây là ý tưởng **counting sort**: khi khoá là số nguyên nhỏ, đánh chỉ số trực tiếp thay cho sort.

In [14]:
def display_bucket(orders):
    foods = sorted({f for _, _, f in orders})
    idx = {f: j for j, f in enumerate(foods)}
    MAXT = 500
    grid = [[0] * len(foods) for _ in range(MAXT + 1)]
    used = [False] * (MAXT + 1)
    for _, t, f in orders:
        ti = int(t)
        grid[ti][idx[f]] += 1
        used[ti] = True
    res = [["Table"] + foods]
    for ti in range(1, MAXT + 1):
        if used[ti]:
            res.append([str(ti)] + [str(c) for c in grid[ti]])
    return res

for orders, want in examples:
    print(display_bucket(orders) == want)

True
True
True


In [15]:
class SolutionBucket:
    def displayTable(self, orders: List[List[str]]) -> List[List[str]]:
        foods = sorted({f for _, _, f in orders})
        idx = {f: j for j, f in enumerate(foods)}
        grid = [[0] * len(foods) for _ in range(501)]
        used = [False] * 501
        for _, t, f in orders:
            ti = int(t)
            grid[ti][idx[f]] += 1
            used[ti] = True
        res = [["Table"] + foods]
        for ti in range(1, 501):
            if used[ti]:
                res.append([str(ti)] + [str(c) for c in grid[ti]])
        return res

show(SolutionBucket().displayTable(ex1))

Table | Beef Burrito | Ceviche | Fried Chicken | Water
3     | 0            | 2       | 1             | 0    
5     | 0            | 1       | 0             | 1    
10    | 1            | 0       | 0             | 0    


**Độ phức tạp:** `O(N + F log F + 500 · F)` thời gian, `O(500 · F)` bộ nhớ. Không phụ thuộc sort bàn; hợp khi số bàn bị chặn nhỏ. Lưu ý `str(int(t))` chuẩn hoá số bàn — đề bảo đảm số bàn hợp lệ nên không có `"03"`.

## 8. TRACE từng bước

Trace cách 2 trên ví dụ 1 — sau mỗi lượt gọi in trạng thái `cnt`:

| lượt | (bàn, món) | `cnt` sau lượt |
|---|---|---|
| 1 | (3, Ceviche) | `3:{Ceviche:1}` |
| 2 | (10, Beef Burrito) | `3:{Ceviche:1}, 10:{Beef Burrito:1}` |
| 3 | (3, Fried Chicken) | `3:{Ceviche:1, Fried Chicken:1}, ...` |
| 4 | (5, Water) | `..., 5:{Water:1}` |
| 5 | (5, Ceviche) | `..., 5:{Water:1, Ceviche:1}` |
| 6 | (3, Ceviche) | `3:{Ceviche:2, Fried Chicken:1}, ...` |

In [16]:
def trace_display(orders):
    cnt = defaultdict(Counter)
    foods = set()
    for i, (name, t, f) in enumerate(orders, 1):
        cnt[t][f] += 1
        foods.add(f)
        state = "; ".join(f"{tb}:{dict(c)}" for tb, c in cnt.items())
        print(f"lượt {i}: khách={name:7s} bàn={t:>2s} món={f:14s} -> {state}")
    foods = sorted(foods)
    print("\nmón đã sắp :", foods)
    order_t = sorted(cnt, key=int)
    print("bàn đã sắp :", order_t, "(theo số, không phải", sorted(cnt), ")")
    res = [["Table"] + foods]
    for t in order_t:
        row = [t] + [str(cnt[t][f]) for f in foods]
        print(f"hàng bàn {t}: {row}")
        res.append(row)
    return res

_ = trace_display(ex1)

lượt 1: khách=David   bàn= 3 món=Ceviche        -> 3:{'Ceviche': 1}
lượt 2: khách=Corina  bàn=10 món=Beef Burrito   -> 3:{'Ceviche': 1}; 10:{'Beef Burrito': 1}
lượt 3: khách=David   bàn= 3 món=Fried Chicken  -> 3:{'Ceviche': 1, 'Fried Chicken': 1}; 10:{'Beef Burrito': 1}
lượt 4: khách=Carla   bàn= 5 món=Water          -> 3:{'Ceviche': 1, 'Fried Chicken': 1}; 10:{'Beef Burrito': 1}; 5:{'Water': 1}
lượt 5: khách=Carla   bàn= 5 món=Ceviche        -> 3:{'Ceviche': 1, 'Fried Chicken': 1}; 10:{'Beef Burrito': 1}; 5:{'Water': 1, 'Ceviche': 1}
lượt 6: khách=Rous    bàn= 3 món=Ceviche        -> 3:{'Ceviche': 2, 'Fried Chicken': 1}; 10:{'Beef Burrito': 1}; 5:{'Water': 1, 'Ceviche': 1}

món đã sắp : ['Beef Burrito', 'Ceviche', 'Fried Chicken', 'Water']
bàn đã sắp : ['3', '5', '10'] (theo số, không phải ['10', '3', '5'] )
hàng bàn 3: ['3', '0', '2', '1', '0']
hàng bàn 5: ['5', '0', '1', '0', '1']
hàng bàn 10: ['10', '1', '0', '0', '0']


## 9. Code nộp LeetCode

Dùng cách 2 (`defaultdict(Counter)`), rõ ràng và đủ nhanh:

In [17]:
class Solution:
    def displayTable(self, orders: List[List[str]]) -> List[List[str]]:
        cnt = defaultdict(Counter)          # cnt[bàn][món] = số lượt gọi
        foods = set()
        for _, table, food in orders:
            cnt[table][food] += 1
            foods.add(food)
        foods = sorted(foods)               # cột: theo bảng chữ
        res = [["Table"] + foods]
        for table in sorted(cnt, key=int):  # hàng: theo giá trị số
            res.append([table] + [str(cnt[table][f]) for f in foods])
        return res

for orders, want in examples:
    assert Solution().displayTable(orders) == want
print("3 ví dụ: OK")

3 ví dụ: OK


## 10. Vì sao thuật toán đúng

- **Số đếm đúng:** mỗi lượt gọi `i` làm tăng đúng một ô `cnt[orders[i][1]][orders[i][2]]` thêm 1, và không ô nào khác. Sau vòng lặp, `cnt[t][f]` bằng số lượt có bàn `t`, món `f` — đúng định nghĩa ô của bảng. Ô không có lượt nào ⇒ `Counter` trả 0.
- **Đủ cột / đủ hàng:** tập `foods` chứa mọi món xuất hiện (mỗi lượt đều `add`), tập khoá `cnt` chứa mọi bàn xuất hiện. Không món/bàn "thừa", vì chỉ thêm khi có lượt gọi.
- **Thứ tự:** `sorted(foods)` cho thứ tự từ điển; `sorted(..., key=int)` cho thứ tự số tăng. Các số bàn khác nhau là các số nguyên khác nhau (không có số 0 đứng đầu) nên không có hoà.
- **Cách 4:** chỉ số mảng `int(t)` tăng dần chính là thứ tự số; `idx[f]` là vị trí cột trong danh sách món đã sắp ⇒ cùng bảng với cách 2.

## 11. Phân tích Time / Space

Gọi `N = len(orders)`, `T` = số bàn khác nhau (`≤ 500`), `F` = số món khác nhau (`≤ N`), `L ≤ 20` là độ dài chuỗi.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(T · F · N)` | `O(T · F)` (kết quả) |
| 2. Dict lồng | `O(N·L + F log F · L + T log T + T·F)` | `O(T·F)` |
| 3. Counter cặp | như cách 2 | `O(T·F)` |
| 4. Bucket 1..500 | `O(N·L + F log F · L + 500·F)` | `O(500·F)` |

Kích thước **đầu ra** đã là `(T+1)·(F+1)` ô, nên `O(T·F)` là chặn dưới — các cách 2–4 đều tối ưu tới thừa số log.

In [18]:
# Đo trên input lớn đúng ràng buộc: N = 5·10^4, T tới 500, F = 200
rng = random.Random(1418)
foods_pool = ["".join(rng.choice("ABCDEFGHIJ abcdefghij") for _ in range(rng.randint(1, 20))).strip() or "X"
              for _ in range(200)]
big = [[f"cust{i}", str(rng.randint(1, 500)), rng.choice(foods_pool)] for i in range(50000)]
for name, f in [("nested", display_nested), ("pairs", display_pairs), ("bucket", display_bucket)]:
    t = time.perf_counter(); r = f(big); el = time.perf_counter() - t
    print(f"{name:7s}: {el:.3f}s, kích thước bảng {len(r)} x {len(r[0])}")

nested : 0.028s, kích thước bảng 501 x 198
pairs  : 0.041s, kích thước bảng 501 x 198
bucket : 0.014s, kích thước bảng 501 x 198


## 12. Bộ test

1. Ba ví dụ của đề.
2. Edge: một lượt gọi; một bàn gọi nhiều món; nhiều bàn cùng một món; số bàn `1`, `9`, `10`, `100`, `500` (bẫy sort chuỗi); tên món chỉ khác hoa/thường; tên món có dấu cách.
3. Đối chiếu 4 cách + brute force trên dữ liệu ngẫu nhiên (seed cố định, đúng ràng buộc).
4. Kiểm tra tính chất: tổng mọi ô = `N`.

In [19]:
impls = {
    "brute": display_brute, "nested": display_nested, "pairs": display_pairs, "bucket": display_bucket,
    "Solution": lambda o: Solution().displayTable(o),
    "SolPairs": lambda o: SolutionPairs().displayTable(o),
    "SolBucket": lambda o: SolutionBucket().displayTable(o),
}
for name, f in impls.items():
    for orders, want in examples:
        assert f(orders) == want, name

edge_cases = [
    ([["A", "7", "Soda"]], [["Table", "Soda"], ["7", "1"]]),
    ([["A", "2", "Tea"], ["A", "2", "Tea"]], [["Table", "Tea"], ["2", "2"]]),   # cùng khách gọi 2 lần -> 2
    ([["a", "10", "X"], ["b", "9", "X"], ["c", "100", "X"], ["d", "1", "X"], ["e", "500", "X"]],
     [["Table", "X"], ["1", "1"], ["9", "1"], ["10", "1"], ["100", "1"], ["500", "1"]]),
    ([["a", "1", "soda"], ["b", "1", "Soda"]], [["Table", "Soda", "soda"], ["1", "1", "1"]]),
    ([["a", "3", "Fried Chicken"], ["b", "3", "Fried  Chicken"]],
     [["Table", "Fried  Chicken", "Fried Chicken"], ["3", "1", "1"]]),
]
for orders, want in edge_cases:
    for name, f in impls.items():
        assert f(orders) == want, (name, orders, f(orders))
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [20]:
rng = random.Random(2024)
def rand_orders(rng):
    n = rng.randint(1, 60)
    names = ["Ann", "Bob", "Cy", "Dee"]
    foods = ["".join(rng.choice("abAB ") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 6))]
    foods = [f if f.strip() else "Z" for f in foods]
    tables = [str(rng.randint(1, 500)) for _ in range(rng.randint(1, 8))]
    return [[rng.choice(names), rng.choice(tables), rng.choice(foods)] for _ in range(n)]

for _ in range(400):
    orders = rand_orders(rng)
    ref = display_brute(orders)
    for name, f in impls.items():
        assert f(orders) == ref, name
    total = sum(int(x) for row in ref[1:] for x in row[1:])
    assert total == len(orders)
    assert all(isinstance(x, str) for row in ref for x in row)
print("random 400 test + tổng ô = N: OK")

random 400 test + tổng ô = N: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sắp số bàn như chuỗi

In [21]:
def wrong_sort_tables(orders):
    cnt = defaultdict(Counter); foods = set()
    for _, t, f in orders:
        cnt[t][f] += 1; foods.add(f)
    foods = sorted(foods)
    return [["Table"] + foods] + [[t] + [str(cnt[t][f]) for f in foods] for t in sorted(cnt)]   # SAI

got = wrong_sort_tables(ex1)
print("thứ tự bàn sai:", [r[0] for r in got[1:]], "| đúng:", [r[0] for r in out1[1:]])
print("bằng đáp án?", got == out1)

thứ tự bàn sai: ['10', '3', '5'] | đúng: ['3', '5', '10']
bằng đáp án? False


**Vì sao sai:** `sorted(["3","10","5"])` so ký tự đầu `'1' < '3' < '5'` ⇒ `["10","3","5"]`. Đề yêu cầu "numerically increasing".

**Sửa:** `sorted(cnt, key=int)` (hoặc lưu khoá dạng `int` rồi `str()` khi in).

In [22]:
print([r[0] for r in Solution().displayTable(ex1)[1:]])

['3', '5', '10']


### Lỗi 2 — Trả về số nguyên thay vì chuỗi

In [23]:
def wrong_int_cells(orders):
    cnt = defaultdict(Counter); foods = set()
    for _, t, f in orders:
        cnt[t][f] += 1; foods.add(f)
    foods = sorted(foods)
    return [["Table"] + foods] + [[t] + [cnt[t][f] for f in foods] for t in sorted(cnt, key=int)]   # SAI: thiếu str

got = wrong_int_cells(ex3)
print(got, "| bằng đáp án?", got == out3)

[['Table', 'Bean Burrito', 'Beef Burrito', 'Soda'], ['2', 1, 1, 1]] | bằng đáp án? False


**Vì sao sai:** đề yêu cầu `List[List[str]]`; `1 != "1"` trong Python nên so sánh với đáp án thất bại (LeetCode báo Wrong Answer).

**Sửa:** bọc `str(...)` cho từng ô số lượng.

### Lỗi 3 — Đếm theo **khách** (dùng set) thay vì theo **lượt gọi**

In [24]:
def wrong_distinct_customers(orders):
    who = defaultdict(set); foods = set()
    for name, t, f in orders:
        who[(t, f)].add(name)                      # SAI: cùng khách gọi 2 lần chỉ tính 1
        foods.add(f)
    foods = sorted(foods)
    tables = sorted({t for _, t, _ in orders}, key=int)
    return [["Table"] + foods] + [[t] + [str(len(who[(t, f)])) for f in foods] for t in tables]

orders = [["David", "3", "Ceviche"], ["David", "3", "Ceviche"]]
print("sai:", wrong_distinct_customers(orders), "| đúng:", Solution().displayTable(orders))

sai: [['Table', 'Ceviche'], ['3', '1']] | đúng: [['Table', 'Ceviche'], ['3', '2']]


**Vì sao sai:** ô của bảng là "bàn đã gọi **bao nhiêu** món đó", mỗi lượt gọi là một phần. Ví dụ 1 có David và Rous cùng gọi Ceviche ⇒ 2, nếu David gọi hai lần cũng phải là 2.

**Sửa:** đếm `+= 1` theo từng lượt, bỏ qua tên khách.

### Lỗi 4 — Chỉ in những món bàn đó đã gọi (thiếu cột `"0"`)

In [25]:
def wrong_missing_zero(orders):
    cnt = defaultdict(Counter); foods = set()
    for _, t, f in orders:
        cnt[t][f] += 1; foods.add(f)
    foods = sorted(foods)
    res = [["Table"] + foods]
    for t in sorted(cnt, key=int):
        res.append([t] + [str(cnt[t][f]) for f in sorted(cnt[t])])    # SAI: chỉ món của bàn t
    return res

got = wrong_missing_zero(ex1)
for row in got: print(row)
print("bằng đáp án?", got == out1)

['Table', 'Beef Burrito', 'Ceviche', 'Fried Chicken', 'Water']
['3', '2', '1']
['5', '1', '1']
['10', '1']
bằng đáp án? False


**Vì sao sai:** mỗi hàng phải có **đủ** `F + 1` cột, căn theo header. Duyệt `cnt[t]` làm hàng ngắn và lệch cột.

**Sửa:** luôn duyệt danh sách `foods` chung, ô vắng in `"0"`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "**lập bảng / pivot table / thống kê theo 2 chiều**" ⇒ `defaultdict(Counter)` hoặc `Counter` trên tuple.
- Thấy **số dạng chuỗi** cần sắp ⇒ luôn tự hỏi "sort chuỗi hay sort số?" — `key=int`.
- Miền khoá nhỏ (bàn ≤ 500) ⇒ có thể dùng mảng đánh chỉ số trực tiếp thay cho sort.
- Đầu ra là **chuỗi** ⇒ kiểm tra `str()` ở mọi ô, kể cả `"0"`.
- Tên khách là **dữ liệu nhiễu** — đọc kỹ đề để biết cột nào được dùng.
- Test nhanh: số bàn `9, 10, 100`; món chỉ khác hoa/thường; cùng khách gọi lặp.

## 15. Biến thể của bài

1. **Thêm cột tổng** mỗi bàn và **hàng tổng** mỗi món (pivot có margin).
2. **Sắp món không phân biệt hoa thường** (`key=str.lower`, hoà thì theo mã gốc).
3. **Đếm số khách khác nhau** mỗi bàn/món (khi đó dùng `set` là đúng!).
4. **Truy vấn trực tuyến:** thêm lượt gọi và hỏi bảng hiện tại nhiều lần ⇒ duy trì `SortedList` món (Ordered Set) để không sort lại.
5. Bài cùng họ: **LeetCode 1636 Sort Array by Increasing Frequency**, **2491**, **SQL pivot** (bài 1179 Reformat Department Table).

Code biến thể 1:

In [26]:
def display_with_totals(orders):
    base = Solution().displayTable(orders)
    header = base[0] + ["Total"]
    rows = [r + [str(sum(int(x) for x in r[1:]))] for r in base[1:]]
    col_tot = ["Total"] + [str(sum(int(r[j]) for r in rows)) for j in range(1, len(header))]
    return [header] + rows + [col_tot]

show(display_with_totals(ex1))

Table | Beef Burrito | Ceviche | Fried Chicken | Water | Total
3     | 0            | 2       | 1             | 0     | 3    
5     | 0            | 1       | 0             | 1     | 2    
10    | 1            | 0       | 0             | 0     | 1    
Total | 1            | 3       | 1             | 1     | 6    


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Mỗi ô quét lại toàn bộ `orders` | `O(T·F·N)` | `O(T·F)` | Kiểm chứng, dữ liệu rất nhỏ |
| 2. Dict lồng | `cnt[bàn][món] += 1`, sort 2 trục | `O(N + F log F + T log T + T·F)` | `O(T·F)` | Mặc định, dễ đọc |
| 3. Counter cặp | `Counter((bàn, món))` | như 2 | `O(T·F)` | Code ngắn nhất |
| 4. Bucket 1..500 | Mảng hàng theo `int(bàn)`, cột theo `idx[món]` | `O(N + F log F + 500·F)` | `O(500·F)` | Khoá số nhỏ, tránh sort |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().displayTable([["a","2","b"],["c","11","a"],["d","2","a"]])` trả về gì?

### Bài 2 (tính tay)
Với `orders = [["x","5","Tea"],["y","1","tea"],["z","5","Tea"],["w","20","Cake"]]`, hãy viết tay header và các hàng (chú ý chữ hoa/thường và thứ tự số bàn).

### Bài 3 (điền TODO)
Hoàn thiện `table_totals(orders)` trả về dict `{bàn (int): tổng số món bàn đó đã gọi}`.

### Bài 4 (tự viết lại)
Viết lại `display_sorted_pairs(orders)` **không dùng dict**: sort các cặp `(int(bàn), món)` rồi gom nhóm liên tiếp bằng một lượt duyệt.

### Bài 5 (mini challenge)
Viết `display_ci(orders)`: giống đề nhưng tên món **gộp không phân biệt hoa thường** (`"Tea"` và `"tea"` là một món), tên hiển thị là dạng **xuất hiện đầu tiên** trong `orders`, cột sắp theo `lower()`.

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def table_totals(orders):
    totals = {}
    # TODO: với mỗi lượt gọi, cộng 1 vào totals[int(bàn)]
    pass

# Bài 4 — TODO
def display_sorted_pairs(orders):
    # TODO: pairs = sorted((int(t), f) for _, t, f in orders)
    # TODO: duyệt pairs, đếm các đoạn giống nhau liên tiếp, điền vào hàng của bàn
    pass

# Bài 5 — TODO
def display_ci(orders):
    # TODO: canon = {} ánh xạ f.lower() -> tên hiển thị đầu tiên
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Món: `a`, `b`. (2) Bàn: `2`, `11` (số!). (3) Bàn 2 gọi `a` 1, `b` 1; bàn 11 gọi `a` 1 ⇒ `[["Table","a","b"],["2","1","1"],["11","1","0"]]`.

**Bài 2.** (1) Món khác nhau: `Cake`, `Tea`, `tea` (chữ hoa trước chữ thường). (2) Bàn: `1`, `5`, `20`. (3) `["1","0","0","1"]`, `["5","0","2","0"]`, `["20","1","0","0"]`.

**Bài 3.** (1) `totals[int(t)] = totals.get(int(t), 0) + 1`. (2) Hoặc `Counter(int(t) for _, t, _ in orders)`. (3) Kiểm tra `sum(totals.values()) == len(orders)`.

**Bài 4.** (1) Sort cặp ⇒ cùng bàn liền nhau, trong bàn món theo thứ tự. (2) Khi cặp đổi, ghi số đếm vào `row[idx[f]]`. (3) Khi bàn đổi, đẩy hàng cũ vào kết quả.

**Bài 5.** (1) Duyệt lần đầu: `canon.setdefault(f.lower(), f)`. (2) Đếm `cnt[t][f.lower()]`. (3) Cột sắp theo khoá `lower` (hoà không xảy ra vì khoá đã gộp), header dùng `canon[key]`.

## 19. Đáp án tham khảo

In [28]:
# Bài 1
assert Solution().displayTable([["a","2","b"],["c","11","a"],["d","2","a"]]) == \
    [["Table","a","b"],["2","1","1"],["11","1","0"]]

# Bài 2
assert Solution().displayTable([["x","5","Tea"],["y","1","tea"],["z","5","Tea"],["w","20","Cake"]]) == \
    [["Table","Cake","Tea","tea"],["1","0","0","1"],["5","0","2","0"],["20","1","0","0"]]

# Bài 3
def table_totals_sol(orders):
    totals = {}
    for _, t, _ in orders:
        totals[int(t)] = totals.get(int(t), 0) + 1
    return totals
assert table_totals_sol(ex1) == {3: 3, 10: 1, 5: 2}

# Bài 4
def display_sorted_pairs_sol(orders):
    pairs = sorted((int(t), f) for _, t, f in orders)
    foods = sorted({f for _, f in pairs})
    idx = {f: j for j, f in enumerate(foods)}
    res = [["Table"] + foods]
    i = 0
    while i < len(pairs):
        t = pairs[i][0]
        row = [0] * len(foods)
        while i < len(pairs) and pairs[i][0] == t:
            j = i
            while j < len(pairs) and pairs[j] == pairs[i]:
                j += 1
            row[idx[pairs[i][1]]] = j - i
            i = j
        res.append([str(t)] + [str(c) for c in row])
    return res

# Bài 5
def display_ci_sol(orders):
    canon = {}
    cnt = defaultdict(Counter)
    for _, t, f in orders:
        key = f.lower()
        canon.setdefault(key, f)
        cnt[int(t)][key] += 1
    keys = sorted(canon)
    res = [["Table"] + [canon[k] for k in keys]]
    for t in sorted(cnt):
        res.append([str(t)] + [str(cnt[t][k]) for k in keys])
    return res

assert display_ci_sol([["x","5","Tea"],["y","1","tea"],["z","5","Tea"],["w","20","Cake"]]) == \
    [["Table","Cake","Tea"],["1","0","1"],["5","0","2"],["20","1","0"]]

rng = random.Random(99)
for _ in range(300):
    orders = rand_orders(rng)
    assert display_sorted_pairs_sol(orders) == Solution().displayTable(orders)
    assert table_totals_sol(orders) == {int(r[0]): sum(map(int, r[1:])) for r in Solution().displayTable(orders)[1:]}
    ci = display_ci_sol(orders)
    assert sum(int(x) for r in ci[1:] for x in r[1:]) == len(orders)
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "pivot table" 2 chiều:**

```python
from collections import defaultdict, Counter

cnt = defaultdict(Counter)        # cnt[hang][cot] = số lần
cols = set()
for rec in data:
    r, c = key_row(rec), key_col(rec)
    cnt[r][c] += 1
    cols.add(c)
cols = sorted(cols)                              # thứ tự cột
rows = sorted(cnt, key=khoa_sap_hang)            # vd key=int
table = [["Header"] + cols] + [[r] + [str(cnt[r][c]) for c in cols] for r in rows]
```

**Checklist:**
- [ ] Header bắt đầu bằng đúng chuỗi `"Table"`.
- [ ] Món sắp theo `sorted()` mặc định (chữ hoa trước chữ thường).
- [ ] Bàn sắp theo **số**: `key=int`.
- [ ] Mọi ô là **chuỗi**, có `"0"` cho món không gọi.
- [ ] Đếm theo lượt gọi, bỏ qua tên khách.
- [ ] Đã thử bàn `9/10/100`, món khác hoa/thường, khách gọi lặp.